# AP Analytics: Unstructured Data Extension

This notebook builds **vendor contract documents** and a **Cortex Search Service** to complement the existing structured AP analytics stack (`AP_INVOICES` + `VENDORS` + `SV_AP_ANALYTICS`).

After running all cells, you'll manually add the search tool to `AP_ANALYTICS_ASSISTANT` so the agent can answer questions that span both structured invoice data and unstructured contract text in a single conversation.

**What gets created:**
- `VENDOR_CONTRACTS` table (~50 rows) with full contract text including payment terms, SLAs, volume discounts, penalties, and renewal clauses
- `VENDOR_CONTRACT_SEARCH` Cortex Search Service over the contract content

In [ ]:
-- Create the vendor contracts table
CREATE OR REPLACE TABLE COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACTS (
    DOC_ID          VARCHAR NOT NULL,
    VENDOR_ID       NUMBER(2,0) NOT NULL,
    CONTRACT_TITLE  VARCHAR NOT NULL,
    EFFECTIVE_DATE  DATE NOT NULL,
    EXPIRATION_DATE DATE NOT NULL,
    CONTENT         VARCHAR NOT NULL
);

In [ ]:
-- Insert vendor contracts (batch 1: Manufacturing vendors)
INSERT INTO COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACTS VALUES
('VC-001', 1, 'Siemens AG - Master Supply Agreement', '2024-01-15', '2026-12-31',
'MASTER SUPPLY AGREEMENT between Buyer and Siemens AG, effective January 15, 2024.

PAYMENT TERMS: All invoices shall be paid NET60 from date of invoice receipt. Early payment discount of 2% is available for payments made within 15 days. Late payments shall accrue interest at 1.5% per month on outstanding balances.

SERVICE LEVEL AGREEMENT: Siemens AG guarantees delivery of ordered components within 14 business days of purchase order acceptance. On-time delivery rate shall meet or exceed 95% measured quarterly. Defect rate on delivered materials shall not exceed 0.5% by volume.

VOLUME DISCOUNTS: Annual spend tiers apply as follows: Tier 1 (EUR 0 - 500,000): standard catalog pricing. Tier 2 (EUR 500,001 - 1,000,000): 5% discount on catalog prices. Tier 3 (EUR 1,000,001+): 8% discount on catalog prices plus dedicated account manager.

PENALTY CLAUSES: If on-time delivery falls below 90% in any quarter, Buyer may apply a 3% penalty on that quarters invoiced amount. If defect rate exceeds 1%, Siemens AG shall replace defective materials at no charge and credit 5% of the affected shipment value.

RENEWAL: This agreement auto-renews for successive 12-month periods unless either party provides 120 days written notice prior to expiration. Governing law: Germany.'),

('VC-002', 1, 'Siemens AG - Engineering Services Addendum', '2025-03-01', '2027-02-28',
'ENGINEERING SERVICES ADDENDUM to Master Supply Agreement between Buyer and Siemens AG.

SCOPE: Siemens AG shall provide on-site engineering consultation for production line optimization projects. Services are billed on a time-and-materials basis.

RATE CARD: Senior Engineer: EUR 185/hour. Project Engineer: EUR 145/hour. Technical Specialist: EUR 120/hour. Minimum engagement: 40 hours per project.

PAYMENT TERMS: Engineering services invoices are due NET60 consistent with the Master Supply Agreement. Travel and accommodation expenses are billed at cost with receipts, capped at EUR 500 per day.

INTELLECTUAL PROPERTY: All designs, specifications, and process improvements developed during engagements are jointly owned. Neither party may license joint IP to third parties without written consent.

CONFIDENTIALITY: Both parties agree to maintain confidentiality of proprietary information for 5 years following contract termination.'),

('VC-003', 2, 'Bosch Zulieferer KG - Component Supply Contract', '2024-06-01', '2026-05-31',
'COMPONENT SUPPLY CONTRACT between Buyer and Bosch Zulieferer KG, effective June 1, 2024.

PAYMENT TERMS: NET45 from invoice date. Bosch Zulieferer KG offers a 1.5% early payment discount for settlement within 10 days.

DELIVERY: Standard lead time is 10 business days for catalog items and 21 business days for custom-machined parts. Expedited delivery available at 15% surcharge with 5 business day lead time.

QUALITY STANDARDS: All components must meet ISO 9001:2015 certification requirements. Incoming inspection reject rate must not exceed 0.3%. Bosch Zulieferer KG maintains a zero-tolerance policy on safety-critical component defects.

VOLUME COMMITMENT: Buyer commits to minimum annual purchase of EUR 200,000. Failure to meet minimum triggers a 2% price adjustment on subsequent orders. Volume above EUR 500,000 qualifies for 6% rebate on annual spend.

WARRANTY: 24-month warranty on all components from date of delivery. Warranty covers material defects and workmanship but excludes damage from misuse or improper installation. Governing law: Germany.'),

('VC-004', 3, 'Rheinland Chemie GmbH - Chemical Supply Agreement', '2024-03-01', '2027-02-28',
'CHEMICAL SUPPLY AGREEMENT between Buyer and Rheinland Chemie GmbH.

PAYMENT TERMS: NET30 from date of delivery confirmation. All pricing is in EUR and subject to annual review based on raw material index fluctuations. Price adjustments exceeding 5% require 60 days advance notice.

SAFETY AND COMPLIANCE: Rheinland Chemie GmbH warrants that all chemical products comply with REACH regulation (EC 1907/2006) and CLP regulation (EC 1272/2008). Safety data sheets shall be provided with every shipment. Rheinland Chemie GmbH maintains EUR 10,000,000 product liability insurance.

STORAGE REQUIREMENTS: Buyer shall maintain proper storage conditions as specified in product documentation. Rheinland Chemie GmbH is not liable for product degradation due to improper storage.

VOLUME PRICING: Orders exceeding EUR 50,000 per shipment receive a 4% volume discount. Annual framework orders with fixed quarterly delivery schedules receive an additional 3% planning discount.

TERMINATION: Either party may terminate with 90 days written notice. Buyer is responsible for accepting all confirmed orders placed prior to termination notice. Governing law: Germany.'),

('VC-005', 10, 'Lyon Industrie SA - Manufacturing Parts Agreement', '2025-01-01', '2027-12-31',
'MANUFACTURING PARTS AGREEMENT between Buyer and Lyon Industrie SA.

PAYMENT TERMS: NET45 from invoice date. Lyon Industrie SA accepts payment in EUR only. Wire transfer is the preferred payment method; credit card payments incur a 2.5% processing fee.

DELIVERY AND LOGISTICS: FOB origin, Lyon warehouse. Standard shipping via ground freight, 5-7 business days within EU. Express air freight available at actual cost plus 10% handling fee.

QUALITY ASSURANCE: Lyon Industrie SA operates under ISO 9001 and IATF 16949 automotive quality standards. First article inspection reports provided for all new part numbers. Statistical process control data available upon request.

PRICING: Firm fixed pricing for 12 months from effective date. Annual price reviews in Q4 for following year. Raw material cost pass-through limited to 3% per annum.

MINIMUM ORDER: EUR 5,000 per purchase order. Orders below minimum subject to EUR 250 small order surcharge. Governing law: France.'),

('VC-006', 14, 'Milano Componenti SpA - Precision Parts Contract', '2024-09-01', '2026-08-31',
'PRECISION PARTS CONTRACT between Buyer and Milano Componenti SpA.

PAYMENT TERMS: NET30. Milano Componenti SpA offers 2% discount for payment within 10 days (2/10 NET30). All amounts in EUR.

TOLERANCES AND SPECIFICATIONS: Milano Componenti SpA guarantees dimensional tolerances within +/- 0.01mm for precision-machined parts. Surface finish Ra 0.8 or better unless otherwise specified. 100% inspection on critical dimensions.

LEAD TIMES: Standard parts: 15 business days. Complex assemblies: 25 business days. Prototype and first articles: 30 business days with engineering review.

PENALTY FOR LATE DELIVERY: If delivery is more than 5 business days late, Milano Componenti SpA credits Buyer 1% of order value per business day of delay, capped at 10% of order value.

REJECTION AND RETURNS: Buyer must notify Milano Componenti SpA of quality issues within 30 days of receipt. Rejected parts are returned at Milano Componenti SpAs expense and replaced within 10 business days. Governing law: Italy.'),

('VC-007', 19, 'Wien Praezision GmbH - Tooling and Fixtures Agreement', '2024-04-01', '2026-03-31',
'TOOLING AND FIXTURES AGREEMENT between Buyer and Wien Praezision GmbH.

PAYMENT TERMS: NET45. Tooling projects exceeding EUR 25,000 require 30% deposit upon order, 40% upon design approval, and 30% upon delivery and acceptance.

DESIGN AND ENGINEERING: Wien Praezision GmbH provides full 3D CAD design services. Design review milestones at 30%, 60%, and 90% completion. Buyer has 10 business days to approve each milestone.

TOOLING OWNERSHIP: All tooling produced under this agreement is owned by Buyer. Wien Praezision GmbH shall store tooling at no charge for up to 24 months of inactivity. After 24 months, storage fee of EUR 100 per tool per month applies.

MAINTENANCE: Wien Praezision GmbH provides preventive maintenance on all active tooling at no additional cost. Repair costs for damage due to normal wear are included. Damage from misuse or exceeding rated capacity is billed at actual cost.

WARRANTY: 12-month warranty on tooling from date of acceptance. Warranty covers defects in materials and workmanship. Tool life guarantee: minimum 50,000 cycles under rated conditions. Governing law: Austria.');

In [ ]:
-- Insert vendor contracts (batch 2: IT Services vendors)
INSERT INTO COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACTS VALUES
('VC-008', 5, 'Frankfurt IT Partner GmbH - Managed Services Agreement', '2024-07-01', '2026-06-30',
'MANAGED IT SERVICES AGREEMENT between Buyer and Frankfurt IT Partner GmbH.

PAYMENT TERMS: NET30 from invoice date, billed monthly in arrears. Annual contract value: EUR 480,000 (EUR 40,000/month).

SERVICE LEVEL AGREEMENT: Frankfurt IT Partner GmbH guarantees 99.9% uptime for managed infrastructure services measured monthly. Planned maintenance windows (max 4 hours/month) are excluded from uptime calculations. Incident response times: P1 Critical - 15 minutes. P2 High - 1 hour. P3 Medium - 4 hours. P4 Low - next business day.

PENALTIES FOR SLA BREACH: If monthly uptime falls below 99.9%, Buyer receives service credits: 99.5%-99.9% = 5% monthly fee credit. 99.0%-99.5% = 10% credit. Below 99.0% = 25% credit. If uptime falls below 99.0% for three consecutive months, Buyer may terminate without penalty.

SECURITY: Frankfurt IT Partner GmbH maintains ISO 27001 certification and SOC 2 Type II compliance. Annual penetration testing results shared with Buyer. Data processing agreement per GDPR Article 28 included as Appendix A.

TERMINATION: 12-month initial term, then month-to-month. Termination requires 90 days written notice. Data migration assistance provided for 30 days post-termination at no additional cost. Governing law: Germany.'),

('VC-009', 16, 'Dublin Cloud Ltd - Cloud Infrastructure Contract', '2024-01-01', '2026-12-31',
'CLOUD INFRASTRUCTURE CONTRACT between Buyer and Dublin Cloud Ltd.

PAYMENT TERMS: NET30, billed monthly based on actual usage plus reserved capacity fees. Minimum monthly commitment: EUR 25,000.

UPTIME SLA: Dublin Cloud Ltd guarantees 99.95% availability for production workloads. Availability is measured per-service, per-region, on a monthly basis. Scheduled maintenance windows are communicated 72 hours in advance and excluded from availability calculations.

SERVICE CREDITS: Monthly availability 99.0% - 99.95%: 10% credit on affected service fees. 95.0% - 99.0%: 25% credit. Below 95.0%: 50% credit. Service credits are applied to next months invoice and do not exceed 50% of monthly fees.

DATA SOVEREIGNTY: All Buyer data resides within EU data centers (Ireland primary, Netherlands DR). Dublin Cloud Ltd will not transfer data outside the EU without explicit written consent. Annual data residency audit reports provided.

RESERVED CAPACITY: 1-year reserved instances receive 20% discount. 3-year reserved instances receive 35% discount. Reserved capacity is non-refundable but transferable between workloads within the same account.

DISASTER RECOVERY: RPO (Recovery Point Objective): 1 hour. RTO (Recovery Time Objective): 4 hours. Quarterly DR testing included at no additional cost. Governing law: Ireland.'),

('VC-010', 20, 'Helsinki Data Oy - Data Platform Services Agreement', '2024-04-01', '2027-03-31',
'DATA PLATFORM SERVICES AGREEMENT between Buyer and Helsinki Data Oy.

PAYMENT TERMS: NET45 from invoice date. Annual contract value approximately EUR 720,000, billed monthly at EUR 60,000 base plus variable usage fees.

UPTIME GUARANTEE: Helsinki Data Oy guarantees 99.99% platform availability measured on a monthly basis. This equates to no more than 4.3 minutes of unplanned downtime per month. Planned maintenance is performed during a 2-hour window on the first Sunday of each month and excluded from availability calculations.

PENALTY FOR DOWNTIME: For each full hour of unplanned downtime below the 99.99% threshold, Buyer receives a credit equal to 2% of that months base fee. Credits are capped at 100% of the monthly base fee. If availability falls below 99.9% in any month, Buyer may terminate with 30 days notice.

DATA PROCESSING: Helsinki Data Oy processes data exclusively within EU jurisdiction. All data at rest encrypted with AES-256. All data in transit encrypted with TLS 1.3. Key management via customer-managed keys (BYOK) available at no additional cost.

SUPPORT TIERS: Standard support (included): 8x5 business hours, 4-hour response. Premium support (+15% of base fee): 24x7, 30-minute response for P1 incidents. Dedicated account manager assigned for contracts exceeding EUR 500,000 annually.

RENEWAL: Auto-renews for 12-month periods unless either party gives 60 days notice. Price increases capped at 3% per renewal term. Governing law: Finland.'),

('VC-011', 29, 'Metro IT Solutions LLC - IT Support Services Contract', '2025-01-01', '2026-12-31',
'IT SUPPORT SERVICES CONTRACT between Buyer and Metro IT Solutions LLC.

PAYMENT TERMS: NET30 from invoice date, billed monthly. Base monthly fee: USD 35,000 for up to 500 support tickets. Overage: USD 85 per additional ticket.

SCOPE OF SERVICES: Metro IT Solutions provides Level 1-3 technical support for Buyers enterprise applications, desktop support, and network infrastructure. Services include: helpdesk staffing (8am-6pm EST weekdays), remote troubleshooting, on-site support within 4 hours for hardware issues, patch management, and quarterly security assessments.

PERFORMANCE METRICS: First call resolution rate: minimum 70%. Average ticket resolution time: P1 - 2 hours, P2 - 8 hours, P3 - 24 hours, P4 - 72 hours. Customer satisfaction score: minimum 4.2 out of 5.0 measured quarterly.

STAFFING: Metro IT Solutions maintains a dedicated team of 8 FTEs for this engagement. Key personnel changes require 30 days advance notice to Buyer. Background checks required for all staff with access to Buyer systems.

PENALTIES: If first call resolution falls below 60% or customer satisfaction below 3.8 for two consecutive quarters, Buyer may reduce monthly fee by 10% until metrics recover. Governing law: State of New York, United States.'),

('VC-012', 39, 'Pinnacle IT Services LLC - Cybersecurity Services Agreement', '2024-10-01', '2026-09-30',
'CYBERSECURITY SERVICES AGREEMENT between Buyer and Pinnacle IT Services LLC.

PAYMENT TERMS: NET45 from invoice date. Annual contract value: USD 600,000 billed quarterly at USD 150,000.

SERVICES: Pinnacle IT provides 24x7 Security Operations Center (SOC) monitoring, threat intelligence, vulnerability management, incident response, and compliance reporting. Services include: continuous monitoring of up to 5,000 endpoints, weekly vulnerability scans, monthly penetration testing of external-facing assets, and annual red team exercise.

INCIDENT RESPONSE SLA: Detection to notification: 15 minutes for critical threats. Containment initiated: 1 hour for critical, 4 hours for high severity. Full incident report: within 24 hours of containment.

COMPLIANCE SUPPORT: Pinnacle IT assists with SOC 2 Type II, ISO 27001, and GDPR compliance evidence gathering. Quarterly compliance dashboards provided. Annual audit support included (up to 40 hours of auditor interface time).

INSURANCE: Pinnacle IT maintains USD 10,000,000 cyber liability insurance and USD 5,000,000 professional liability (E&O) insurance.

TERMINATION: 24-month initial term. Early termination fee: 50% of remaining contract value. Post-termination transition assistance: 60 days at standard rates. Governing law: State of Delaware, United States.');

In [ ]:
-- Insert vendor contracts (batch 3: Consulting vendors)
INSERT INTO COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACTS VALUES
('VC-013', 9, 'Paris Conseil SARL - Strategic Consulting Framework', '2024-02-01', '2026-01-31',
'STRATEGIC CONSULTING FRAMEWORK AGREEMENT between Buyer and Paris Conseil SARL.

PAYMENT TERMS: NET45 from invoice date. Invoices submitted bi-weekly with detailed time and expense reports. All amounts in EUR.

RATE CARD: Managing Director: EUR 350/hour. Senior Consultant: EUR 250/hour. Consultant: EUR 175/hour. Analyst: EUR 120/hour. Blended rate for team engagements: EUR 210/hour. Rates fixed for 12 months from effective date; annual adjustment capped at CPI + 1%.

ENGAGEMENT TYPES: Time and Materials (T&M): billed at actual hours per rate card. Fixed Price: agreed upon per statement of work, with milestone-based payments (typically 25% at kickoff, 50% at midpoint deliverable, 25% at final acceptance). Retainer: monthly fixed fee for guaranteed availability of named consultants.

TRAVEL AND EXPENSES: Travel billed at cost with prior approval. Air travel: economy class for flights under 4 hours, business class permitted for longer flights. Hotel: capped at EUR 250/night. Meals: capped at EUR 75/day per consultant.

INTELLECTUAL PROPERTY: All work product created during engagements is owned by Buyer upon payment. Paris Conseil SARL retains the right to use general methodologies and frameworks (not client-specific data) in other engagements.

NON-SOLICITATION: Neither party shall solicit or hire employees of the other for 12 months following engagement completion. Governing law: France.'),

('VC-014', 18, 'Bruxelles Conseil SA - Regulatory Compliance Advisory', '2024-08-01', '2026-07-31',
'REGULATORY COMPLIANCE ADVISORY AGREEMENT between Buyer and Bruxelles Conseil SA.

PAYMENT TERMS: NET30 from invoice date. Retainer model: EUR 45,000 per month for up to 200 advisory hours. Hours beyond 200 billed at EUR 275/hour.

SCOPE: Bruxelles Conseil SA provides regulatory compliance advisory services covering EU trade regulations, customs procedures, VAT compliance across multiple EU jurisdictions, REACH and environmental compliance, and corporate governance requirements for EU subsidiaries.

DELIVERABLES: Monthly regulatory update briefing. Quarterly compliance risk assessment report. Annual comprehensive compliance audit with remediation roadmap. Ad-hoc regulatory impact assessments for new business initiatives (within retainer hours).

QUALIFICATIONS: All advisors hold relevant professional certifications (CPA, ACCA, or equivalent). Lead advisor has minimum 15 years EU regulatory experience. Bruxelles Conseil SA maintains professional indemnity insurance of EUR 5,000,000.

CONFIDENTIALITY: Strict confidentiality for 7 years post-termination. Bruxelles Conseil SA will not provide services to direct competitors without Buyers written consent during the contract term.

TERMINATION: Either party may terminate with 60 days notice. On termination, Bruxelles Conseil SA provides complete knowledge transfer documentation within 30 days. Governing law: Belgium.'),

('VC-015', 30, 'Beacon Consulting Group - Management Consulting MSA', '2024-05-01', '2026-04-30',
'MANAGEMENT CONSULTING MASTER SERVICE AGREEMENT between Buyer and Beacon Consulting Group.

PAYMENT TERMS: NET30 from invoice date. Invoices submitted monthly with detailed time reports approved by Buyer project manager.

RATE CARD: Partner: USD 450/hour. Senior Manager: USD 350/hour. Senior Consultant: USD 275/hour. Consultant: USD 200/hour. Associate: USD 150/hour. Weekend and holiday work billed at 1.5x standard rates.

PROJECT GOVERNANCE: Each engagement requires a signed Statement of Work (SOW). SOWs define scope, deliverables, timeline, and budget. Change orders for scope modifications require written approval and may adjust fees and timelines.

PERFORMANCE GUARANTEES: Beacon Consulting Group guarantees that all deliverables meet the acceptance criteria defined in the SOW. If deliverables fail acceptance testing, Beacon will remediate at no additional cost within the original timeline plus 15 business days.

KEY PERSONNEL: Named key personnel in each SOW cannot be reassigned without 30 days notice and Buyer approval. Replacement personnel must have equivalent or superior qualifications.

AUTO-RENEWAL: This MSA auto-renews for successive 12-month periods unless either party provides 90 days written notice prior to the current term expiration. Individual SOWs expire per their own terms regardless of MSA renewal. Governing law: State of New York, United States.'),

('VC-016', 40, 'Evergreen Consulting Inc - Digital Transformation Advisory', '2025-02-01', '2027-01-31',
'DIGITAL TRANSFORMATION ADVISORY AGREEMENT between Buyer and Evergreen Consulting Inc.

PAYMENT TERMS: NET45 from invoice date. Fixed-price engagement: USD 1,200,000 total, payable in 12 monthly installments of USD 100,000.

SCOPE: Evergreen Consulting provides end-to-end digital transformation advisory services including: technology roadmap development, cloud migration strategy, data analytics maturity assessment, AI/ML use case identification, change management planning, and vendor selection support.

DELIVERABLES: Phase 1 (months 1-3): Current state assessment and gap analysis. Phase 2 (months 4-8): Future state architecture and implementation roadmap. Phase 3 (months 9-12): Pilot implementation support and knowledge transfer.

TEAM: Evergreen commits a dedicated team of 4 consultants (1 Principal, 1 Senior, 2 Consultants) for the engagement duration. Team members allocated at minimum 60% to this engagement.

SUCCESS METRICS: Project success measured against KPIs defined in Phase 1, including: time-to-insight reduction of 40%, operational efficiency improvement of 25%, and user adoption rate of 80% for new tools.

RISK MANAGEMENT: Bi-weekly steering committee meetings. Monthly risk register review. Escalation path: Project Manager to Engagement Partner to Managing Director within 48 hours. Governing law: State of California, United States.');

In [ ]:
-- Insert vendor contracts (batch 4: Logistics vendors)
INSERT INTO COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACTS VALUES
('VC-017', 6, 'Global Logistik BV - Freight Services Agreement', '2024-05-01', '2026-04-30',
'FREIGHT SERVICES AGREEMENT between Buyer and Global Logistik BV.

PAYMENT TERMS: NET30 from invoice date. Invoices generated per shipment with consolidated monthly summary. All amounts in EUR.

TRANSIT TIMES: Intra-EU ground freight: 3-5 business days. EU to UK: 5-7 business days (includes customs clearance). Expedited service: next business day within Benelux, 2 business days EU-wide, at 200% standard rate.

DAMAGE AND LOSS LIABILITY: Global Logistik BV maintains cargo insurance of EUR 500,000 per shipment. Liability for damage or loss: full replacement value up to EUR 100,000 per claim. Claims must be filed within 14 days of delivery.

FUEL SURCHARGE: Variable fuel surcharge applied monthly based on EU diesel index. Current cap: 6% of base freight rate. Surcharge adjusted on the 1st of each month with 5 business days advance notice.

VOLUME DISCOUNTS: Monthly freight spend EUR 0-20,000: standard rates. EUR 20,001-50,000: 5% discount. EUR 50,001-100,000: 10% discount. EUR 100,001+: 15% discount plus dedicated logistics coordinator.

SUSTAINABILITY: Global Logistik BV commits to carbon-neutral shipping options. CO2 offset certificate provided for each shipment at no additional cost. Electric vehicle fleet for last-mile delivery in major EU cities by 2025. Governing law: Netherlands.'),

('VC-018', 7, 'Amsterdam Freight NV - Warehousing and Distribution', '2024-03-01', '2026-02-28',
'WAREHOUSING AND DISTRIBUTION AGREEMENT between Buyer and Amsterdam Freight NV.

PAYMENT TERMS: NET45 from monthly invoice date. Monthly storage fees billed in advance; distribution fees billed in arrears based on actual activity.

STORAGE RATES: Pallet storage: EUR 8.50 per pallet position per week. Bulk storage: EUR 4.50 per square meter per week. Temperature-controlled storage: EUR 14.00 per pallet position per week. Hazardous materials storage: EUR 22.00 per pallet position per week (subject to regulatory surcharges).

DISTRIBUTION SLA: Order received by 2pm CET ships same day. Next-day delivery guaranteed within Benelux. 2-3 day delivery within EU. Order accuracy target: 99.7%. Pick-and-pack error rate: maximum 0.1%.

INVENTORY MANAGEMENT: Amsterdam Freight NV provides real-time inventory visibility via web portal. Cycle counting performed monthly. Annual physical inventory count included at no additional cost. Inventory discrepancy tolerance: 0.5% by value.

INSURANCE: Warehouse legal liability insurance: EUR 2,000,000. All-risk cargo insurance during storage and transit: EUR 1,000,000 per occurrence. Governing law: Netherlands.'),

('VC-019', 23, 'Pacific Freight LLC - US Domestic Freight Contract', '2024-08-01', '2026-07-31',
'US DOMESTIC FREIGHT CONTRACT between Buyer and Pacific Freight LLC.

PAYMENT TERMS: NET30 from invoice date. All amounts in USD. Electronic invoicing via EDI 810 required.

TRANSIT TIMES: West Coast to East Coast (ground): 5-7 business days. Regional (within same coast): 2-3 business days. LTL (Less Than Truckload): add 1-2 business days to standard transit. Expedited: guaranteed next-day for shipments tendered by 12pm local time.

FUEL SURCHARGE: Variable surcharge based on US DOE National Average Diesel Price. Current cap: 8% of base freight rate. Surcharge recalculated weekly, posted every Monday for the following week.

VOLUME DISCOUNTS: Monthly freight spend USD 0-25,000: standard tariff rates. USD 25,001-50,000: 8% discount. USD 50,001-100,000: 12% discount. USD 100,001+: 15% discount plus quarterly business reviews with dedicated account executive.

CLAIMS: Freight claims must be filed within 9 months of delivery per Carmack Amendment. Pacific Freight LLC liability limited to USD 25 per pound unless higher value declared at time of shipment. Declared value surcharge: 2% of declared amount.

ACCESSORIAL CHARGES: Liftgate delivery: USD 75. Inside delivery: USD 150. Residential delivery: USD 95. Detention (after 2 free hours): USD 75/hour. Governing law: State of California, United States.'),

('VC-020', 11, 'Marseille Transport SAS - Mediterranean Shipping Contract', '2025-01-01', '2026-12-31',
'MEDITERRANEAN SHIPPING CONTRACT between Buyer and Marseille Transport SAS.

PAYMENT TERMS: NET30 from bill of lading date. All amounts in EUR.

ROUTES: Regular service: Marseille to Barcelona (3 days), Marseille to Genoa (2 days), Marseille to Tunis (4 days), Marseille to Istanbul (7 days). Frequency: weekly sailings on fixed schedule.

CONTAINER RATES: 20ft standard container: EUR 1,200-2,800 depending on route. 40ft standard container: EUR 1,800-4,200 depending on route. Refrigerated containers: 40% premium over standard rates.

BOOKING GUARANTEE: Confirmed bookings guaranteed space on scheduled vessel. Carrier cancellation penalty: EUR 500 per TEU. Buyer cancellation within 48 hours of sailing: 25% of freight rate.

DEMURRAGE AND DETENTION: Free time: 7 calendar days at port of discharge. Demurrage after free time: EUR 75 per container per day for first 7 days, EUR 150 per day thereafter. Governing law: France.'),

('VC-021', 37, 'Gateway Logistics Inc - Last-Mile Delivery Contract', '2024-11-01', '2026-10-31',
'LAST-MILE DELIVERY CONTRACT between Buyer and Gateway Logistics Inc.

PAYMENT TERMS: NET30. Monthly minimum commitment: USD 15,000.

DELIVERY SLA: Same-day delivery for orders placed before 10am local time within metro areas. Next-day delivery for all other US locations in the lower 48 states. Saturday delivery available at 25% premium.

ON-TIME DELIVERY TARGET: 98% on-time delivery rate measured monthly. If rate falls below 95% for any month, Gateway Logistics credits 5% of that months fees. Below 90%: 10% credit.

PROOF OF DELIVERY: Electronic proof of delivery (ePOD) with GPS timestamp, signature capture, and photo documentation. ePOD available in real-time via API integration or web portal.

RETURNS HANDLING: Gateway Logistics manages return pickups at USD 12 per package (standard) or USD 8 per package for pre-scheduled routes. Return processing and quality inspection: USD 5 per unit. Governing law: State of Illinois, United States.');

In [ ]:
-- Insert vendor contracts (batch 5: Office Supplies, Facilities, Marketing, Utilities, Travel, Professional Services)
INSERT INTO COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACTS VALUES
('VC-022', 21, 'Acme Supply Co - Office Supplies Master Agreement', '2024-01-01', '2025-12-31',
'OFFICE SUPPLIES MASTER AGREEMENT between Buyer and Acme Supply Co.

PAYMENT TERMS: NET30 from invoice date. Purchase card (P-Card) payments accepted with no surcharge.

CATALOG PRICING: Acme Supply Co provides a customized catalog of 5,000+ office supply items at 15% below manufacturer suggested retail price (MSRP). Catalog refreshed quarterly. Price match guarantee: Acme will match any competitors published price on identical items.

MINIMUM ORDER: USD 100 per order for free delivery. Orders below USD 100 subject to USD 12.95 delivery fee. No minimum order for P-Card purchases picked up at local Acme branch.

GREEN PRODUCTS: Minimum 30% of catalog items are FSC-certified or made from recycled materials. Acme Supply Co provides quarterly sustainability reports on Buyers purchasing patterns.

RETURNS: Unused items in original packaging may be returned within 60 days for full credit. Opened items accepted within 30 days for exchange only (no refund). Custom-printed items are non-returnable.

DEDICATED SUPPORT: Named account representative for orders and issue resolution. Online ordering portal with budget tracking and approval workflows. Monthly usage reports by department. Governing law: State of Texas, United States.'),

('VC-023', 28, 'Sierra Packaging Inc - Packaging Materials Contract', '2024-06-01', '2026-05-31',
'PACKAGING MATERIALS CONTRACT between Buyer and Sierra Packaging Inc.

PAYMENT TERMS: NET45 from invoice date. Volume rebates calculated and credited annually in January for prior year purchases.

PRODUCT CATEGORIES: Corrugated boxes (standard and custom sizes), protective packaging (bubble wrap, foam inserts, air pillows), shipping labels and tape, pallets and stretch wrap, custom branded packaging.

PRICING: Standard catalog items: fixed pricing for 6 months. Custom packaging: pricing per approved artwork and specification sheet. Volume pricing tiers: USD 0-50,000/year: catalog price. USD 50,001-150,000: 8% discount. USD 150,001+: 12% discount plus custom design services at no additional charge.

LEAD TIMES: Standard catalog items: 3-5 business days. Custom corrugated: 10-15 business days. Custom printed packaging: 15-20 business days (after artwork approval). Rush orders: 50% surcharge, reduces lead time by half.

SUSTAINABILITY COMMITMENT: All corrugated products contain minimum 70% recycled content. Sierra Packaging commits to 100% recyclable or compostable packaging options by 2026. FSC Chain of Custody certification maintained. Governing law: State of Nevada, United States.'),

('VC-024', 36, 'Empire Office Products - Furniture and Equipment Agreement', '2025-03-01', '2027-02-28',
'FURNITURE AND EQUIPMENT AGREEMENT between Buyer and Empire Office Products.

PAYMENT TERMS: NET60 for standard catalog purchases. Capital equipment orders exceeding USD 25,000: 50% upon order, 50% upon delivery and installation.

PRODUCT SCOPE: Office furniture (desks, chairs, conference tables, storage), ergonomic equipment (standing desks, monitor arms, keyboard trays), office technology (printers, copiers, AV equipment for conference rooms).

INSTALLATION: Empire Office Products provides professional installation for all furniture orders exceeding USD 5,000. Standard installation within 10 business days of delivery. After-hours and weekend installation available at 25% premium.

WARRANTY: Office furniture: 10-year structural warranty, 5-year finish warranty. Ergonomic equipment: 5-year warranty. Office technology: manufacturer warranty pass-through plus Empire 1-year extended coverage. On-site warranty service within 48 hours.

ASSET MANAGEMENT: Empire Office Products provides asset tagging and tracking database for all purchased items. End-of-life disposal and recycling services included at no additional cost for items originally purchased under this agreement. Governing law: State of New York, United States.'),

('VC-025', 15, 'Roma Facility Srl - Facilities Management Contract', '2024-01-01', '2026-12-31',
'FACILITIES MANAGEMENT CONTRACT between Buyer and Roma Facility Srl.

PAYMENT TERMS: NET30 from monthly invoice date. Fixed monthly fee: EUR 85,000 for comprehensive facilities management. Variable costs (materials, subcontractors) billed at cost plus 12% management fee.

SCOPE: Janitorial services (daily), HVAC maintenance (quarterly preventive, emergency response), electrical systems maintenance, plumbing, grounds maintenance, security systems monitoring, reception and mailroom services.

SLA: Emergency response time: 30 minutes on-site during business hours, 2 hours after hours. Preventive maintenance completion rate: 100% per schedule. Facility cleanliness audit score: minimum 90/100 (audited quarterly by independent assessor). Energy efficiency target: 5% annual reduction in energy consumption per square meter.

STAFFING: Roma Facility Srl maintains 25 FTE staff dedicated to Buyers facilities. Site manager with minimum 10 years experience. All staff background-checked and trained per Buyer safety protocols.

INSURANCE: Roma Facility Srl maintains EUR 5,000,000 general liability, EUR 2,000,000 professional liability, and workers compensation coverage per Italian law. Governing law: Italy.'),

('VC-026', 32, 'Capital Facilities Corp - Building Maintenance Contract', '2024-04-01', '2026-03-31',
'BUILDING MAINTENANCE CONTRACT between Buyer and Capital Facilities Corp.

PAYMENT TERMS: NET30 from monthly invoice date. Annual contract value: USD 1,440,000 (USD 120,000/month).

SERVICES: Comprehensive building maintenance including HVAC (preventive and reactive), plumbing, electrical, elevator maintenance coordination, fire protection systems, roof inspections, parking structure maintenance, and common area management.

PERFORMANCE METRICS: Work order completion within SLA: 95%. Preventive maintenance schedule adherence: 98%. Tenant satisfaction survey score: minimum 4.0/5.0. Emergency response: on-site within 1 hour, 24/7/365.

ENERGY MANAGEMENT: Capital Facilities Corp manages building automation systems (BAS) to optimize energy usage. Monthly energy consumption reports. Target: ENERGY STAR score of 75 or above. LED lighting retrofit completion by end of Year 1.

VENDOR MANAGEMENT: Capital Facilities Corp manages all sub-contractor relationships for specialized services (elevator, fire suppression, pest control). Subcontractors must carry minimum USD 2,000,000 liability insurance and be approved by Buyer.

TERMINATION: 30-month initial term. Early termination fee: 6 months of base fees. Transition assistance: 90 days at standard rates. Governing law: State of Virginia, United States.'),

('VC-027', 31, 'Hudson Marketing Partners - Marketing Services Agreement', '2024-09-01', '2026-08-31',
'MARKETING SERVICES AGREEMENT between Buyer and Hudson Marketing Partners.

PAYMENT TERMS: NET30 from invoice date. Retainer: USD 50,000/month for ongoing services. Project work billed separately per approved SOW.

SCOPE: Brand strategy and management, digital marketing (SEO, SEM, social media), content creation (blog, video, white papers), event marketing, market research, analytics and reporting.

PERFORMANCE KPIS: Monthly reporting on: website traffic (target 10% YoY growth), lead generation (target 200 MQLs/month), social media engagement (target 5% engagement rate), email open rate (target 25%+), campaign ROI (target 3:1).

INTELLECTUAL PROPERTY: All creative work product, including designs, copy, and campaign assets, is work-for-hire and owned by Buyer. Hudson Marketing Partners may use work in their portfolio with Buyer written approval.

MEDIA BUYING: Hudson Marketing Partners manages media buying with 15% agency commission on gross media spend. Buyer approves all media plans and budgets in advance. Monthly reconciliation of actual vs. planned spend.

EXCLUSIVITY: Hudson Marketing Partners agrees not to provide services to direct competitors during the contract term plus 6 months post-termination. Governing law: State of New York, United States.'),

('VC-028', 13, 'Barcelona Marketing SL - EU Marketing Campaign Contract', '2025-01-01', '2026-12-31',
'EU MARKETING CAMPAIGN CONTRACT between Buyer and Barcelona Marketing SL.

PAYMENT TERMS: NET45 from invoice date. All amounts in EUR. Campaign budgets approved quarterly in advance.

SCOPE: Pan-European marketing campaigns targeting DACH, Benelux, Iberia, and Nordics markets. Services include: market research, campaign creative development, multi-language content localization (DE, FR, ES, NL, SV), digital advertising management, influencer partnerships, and trade show support.

DELIVERABLES: Quarterly campaign plans with KPIs. Monthly performance dashboards. Bi-annual brand health surveys across target markets. All creative assets in brand-compliant templates.

LOCALIZATION: All content reviewed by native-speaker editors. Cultural sensitivity review for each market. Compliance with local advertising regulations (particularly GDPR consent for digital campaigns).

BUDGET MANAGEMENT: Barcelona Marketing SL manages approved campaign budgets with 10% agency commission on third-party costs. Monthly budget variance reports. Any spend exceeding approved budget by more than 5% requires written pre-approval. Governing law: Spain.'),

('VC-029', 17, 'Shannon Utilities Ltd - Energy Supply Agreement', '2024-06-01', '2027-05-31',
'ENERGY SUPPLY AGREEMENT between Buyer and Shannon Utilities Ltd.

PAYMENT TERMS: NET30 from meter reading date. Monthly billing based on actual consumption.

ELECTRICITY PRICING: Fixed rate: EUR 0.145 per kWh for first 500,000 kWh/month. Excess consumption: EUR 0.135 per kWh (volume discount). Demand charge: EUR 12.50 per kW of peak demand. Green energy premium (100% renewable sourced): additional EUR 0.008 per kWh.

GAS PRICING: Fixed rate for heating season (October-March): EUR 0.065 per kWh. Summer rate (April-September): EUR 0.055 per kWh. Annual consumption commitment: 2,000,000 kWh minimum.

RENEWABLE ENERGY: Shannon Utilities Ltd sources 60% of electricity from wind and solar as of 2024, targeting 80% by 2026. Guarantees of Origin (GoO) certificates provided monthly. Carbon offset program available for remaining fossil fuel consumption.

PRICE REVIEW: Annual price review in September for following calendar year. Maximum annual increase: 5% on fixed rates. Demand charges adjusted based on actual infrastructure costs with 90 days notice.

FORCE MAJEURE: Service interruptions due to grid emergencies, extreme weather, or regulatory orders are excluded from SLA commitments. Shannon Utilities Ltd maintains backup generation capacity for critical facilities. Governing law: Ireland.'),

('VC-030', 33, 'Frontier Utilities Inc - US Energy Services Contract', '2024-01-01', '2026-12-31',
'US ENERGY SERVICES CONTRACT between Buyer and Frontier Utilities Inc.

PAYMENT TERMS: NET30 from meter reading date. Budget billing option available with quarterly true-up.

ELECTRICITY: Fixed rate: USD 0.089 per kWh for 36-month term. No fuel adjustment charges during fixed-rate term. Demand charge: USD 15.00 per kW. Time-of-use pricing available as alternative: peak (2pm-7pm weekdays) USD 0.12/kWh, off-peak USD 0.065/kWh.

DEMAND RESPONSE PROGRAM: Buyer may participate in demand response events (maximum 10 per year, maximum 4 hours each). Compensation: USD 200 per kW of curtailed demand per event. Advance notice: minimum 2 hours before event.

RENEWABLE ENERGY CREDITS (RECs): Frontier Utilities offers REC purchase at USD 2.50 per MWh to offset carbon footprint. Annual sustainability report provided detailing energy sourcing mix and carbon equivalent reductions.

INFRASTRUCTURE: Frontier Utilities responsible for maintenance of utility infrastructure up to the meter. Buyer responsible for all wiring and equipment beyond the meter. Emergency line repairs completed within 4 hours. Governing law: State of Texas, United States.');

In [ ]:
-- Insert vendor contracts (batch 6: Travel, Professional Services, and additional contracts for key vendors)
INSERT INTO COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACTS VALUES
('VC-031', 34, 'Skyline Travel Services - Corporate Travel Management', '2024-03-01', '2026-02-28',
'CORPORATE TRAVEL MANAGEMENT AGREEMENT between Buyer and Skyline Travel Services.

PAYMENT TERMS: NET15 from monthly consolidated invoice. P-Card accepted for individual bookings.

SERVICES: Full-service corporate travel management including: flight bookings, hotel reservations, ground transportation, visa and passport services, travel insurance, 24/7 emergency travel assistance, and expense reporting integration.

PRICING: Transaction fees: domestic flight booking USD 25, international USD 45, hotel USD 15, car rental USD 10. No fees for online self-service bookings through Skyline portal. Monthly management fee: USD 2,500 for dedicated travel desk (2 agents, business hours).

NEGOTIATED RATES: Skyline maintains preferred hotel rates averaging 22% below BAR (Best Available Rate) at 500+ hotels. Airline negotiated fares on top 10 routes averaging 15% savings. Car rental: Enterprise/National preferred rates at 20% below published rates.

DUTY OF CARE: Real-time traveler tracking. Automated alerts for security incidents, weather events, and travel disruptions within 50 miles of any active traveler. Emergency evacuation coordination included.

REPORTING: Monthly travel spend analytics by department, destination, advance booking trends, and policy compliance. Quarterly savings analysis versus market rates. Governing law: State of Colorado, United States.'),

('VC-032', 12, 'Iberia Servicios SL - Professional Services Agreement', '2024-10-01', '2026-09-30',
'PROFESSIONAL SERVICES AGREEMENT between Buyer and Iberia Servicios SL.

PAYMENT TERMS: NET45 from invoice date. All amounts in EUR.

SERVICES: Iberia Servicios provides professional staffing and project management services for Buyers Iberian operations. Services include: project management (PMP-certified managers), business analysis, process improvement (Lean Six Sigma), quality assurance, and regulatory liaison with Spanish authorities.

RATE CARD: Project Manager: EUR 135/hour. Senior Business Analyst: EUR 110/hour. Business Analyst: EUR 85/hour. Quality Assurance Specialist: EUR 95/hour. Regulatory Liaison: EUR 120/hour.

STAFFING GUARANTEE: Iberia Servicios guarantees candidate placement within 15 business days of approved requisition. If placed resource is unsatisfactory, replacement provided within 10 business days at no additional cost for the transition period.

COMPLIANCE: All staff are legally authorized to work in Spain and EU. Iberia Servicios handles all employment obligations including social security, taxes, and labor law compliance. Annual SOC 2 audit report provided.

LIABILITY: Iberia Servicios maintains EUR 3,000,000 professional liability insurance and EUR 2,000,000 general liability insurance. Governing law: Spain.'),

('VC-033', 35, 'Liberty Professional LLC - Legal and Compliance Services', '2024-06-01', '2026-05-31',
'LEGAL AND COMPLIANCE SERVICES AGREEMENT between Buyer and Liberty Professional LLC.

PAYMENT TERMS: NET30 from invoice date. Retainer: USD 25,000/month for ongoing advisory services. Litigation and special projects billed separately per engagement letter.

SCOPE: General corporate legal advisory, contract review and negotiation, employment law compliance, regulatory filings, intellectual property management, and M&A due diligence support.

RATES: Senior Partner: USD 550/hour. Partner: USD 425/hour. Senior Associate: USD 300/hour. Associate: USD 225/hour. Paralegal: USD 125/hour. Retainer hours credited at blended rate of USD 325/hour.

CONFLICTS: Liberty Professional maintains a comprehensive conflicts database and will notify Buyer of any potential conflicts within 5 business days of engagement. If conflict arises during engagement, Liberty assists with transition to alternative counsel.

PRIVILEGE: All communications and work product protected by attorney-client privilege. Liberty Professional maintains strict information barriers between client matters. Document retention per applicable bar rules (minimum 7 years). Governing law: State of Delaware, United States.'),

('VC-034', 4, 'Muller Buerobedarf GmbH - Office Equipment Lease', '2024-11-01', '2027-10-31',
'OFFICE EQUIPMENT LEASE AGREEMENT between Buyer and Muller Buerobedarf GmbH.

PAYMENT TERMS: NET30 from monthly invoice date. Fixed monthly lease payment: EUR 8,500 for all leased equipment.

EQUIPMENT: 50 multifunction printers/copiers (networked, color, scan-to-email), 200 desktop monitors (27" 4K), 30 conference room displays (65" interactive), and associated mounting hardware and cabling.

MAINTENANCE: All-inclusive maintenance and support. On-site technician response within 4 business hours for printer issues, next business day for displays. Toner and consumables for printers included in lease. Paper not included.

REFRESH CYCLE: Equipment refreshed at month 36 with current-generation equivalents at no additional cost. Mid-term upgrades available at prorated differential.

END OF LEASE: At lease expiration, Buyer may: (a) return equipment at no cost, (b) purchase at fair market value (estimated 10-15% of original), (c) renew lease at reduced rate (estimated 40% of original monthly payment).

DATA SECURITY: All returned equipment undergoes certified data destruction (NIST 800-88 compliant). Certificate of destruction provided within 30 days of equipment return. Governing law: Germany.'),

('VC-035', 8, 'Benelux Verpakking NV - Industrial Packaging Supply', '2024-07-01', '2026-06-30',
'INDUSTRIAL PACKAGING SUPPLY AGREEMENT between Buyer and Benelux Verpakking NV.

PAYMENT TERMS: NET30 from delivery date. All amounts in EUR.

PRODUCT RANGE: Heavy-duty corrugated containers (single, double, triple wall), custom foam inserts and protective packaging, industrial stretch wrap and strapping, export-grade pallets (ISPM 15 certified), and anti-corrosion packaging (VCI papers and films).

PRICING: Catalog pricing discounted 18% from published list. Custom items quoted within 3 business days. Annual price adjustment limited to 4% per year. Raw material surcharges (paper, resin) passed through at index when increase exceeds 10%.

QUALITY: All products tested to ISTA 3A transit standards. Corrugated board meeting ECT (Edge Crush Test) specifications per order. Certificate of compliance provided per lot.

DELIVERY: Standard orders: 5-7 business days. Stock items: 2-3 business days. Custom packaging: 10-15 business days after proof approval. Free delivery for orders exceeding EUR 2,500. Governing law: Netherlands.'),

('VC-036', 22, 'Summit Manufacturing Inc - Contract Manufacturing Agreement', '2024-02-01', '2027-01-31',
'CONTRACT MANUFACTURING AGREEMENT between Buyer and Summit Manufacturing Inc.

PAYMENT TERMS: NET45 from shipment date. Tooling and NRE (Non-Recurring Engineering) charges: 50% deposit, 50% on first article approval.

MANUFACTURING SERVICES: Summit Manufacturing provides contract manufacturing services including: CNC machining, sheet metal fabrication, welding and assembly, surface treatment (anodizing, powder coating, plating), and final assembly with testing.

QUALITY SYSTEM: Summit operates under ISO 9001:2015 and AS9100D (aerospace). PPAP (Production Part Approval Process) Level 3 documentation provided for all new parts. In-process inspection at critical operations. Final inspection with CMM (Coordinate Measuring Machine) for dimensional verification.

PRICING: Unit pricing per approved quotation, fixed for 12 months. Raw material price adjustments quarterly based on published indices (steel, aluminum, copper). Labor rate adjustments annually, capped at 3%.

CAPACITY RESERVATION: Buyer guaranteed 20% of Summit production capacity. Capacity increases require 90 days notice. Surge capacity (up to 150% of baseline) available with 30 days notice and 10% premium.

INTELLECTUAL PROPERTY: All Buyer-owned designs and specifications are confidential. Summit will not manufacture identical parts for third parties. Tooling owned by Buyer, stored at Summit at no charge. Governing law: State of Ohio, United States.'),

('VC-037', 24, 'Redwood Materials Corp - Raw Materials Supply Agreement', '2024-04-01', '2026-09-30',
'RAW MATERIALS SUPPLY AGREEMENT between Buyer and Redwood Materials Corp.

PAYMENT TERMS: NET30 from delivery date. Large orders (exceeding USD 100,000): 50% upon order confirmation, 50% upon delivery.

MATERIALS: Specialty metals (aluminum alloys 6061, 7075; stainless steel 304, 316; titanium Grade 5), engineering plastics (PEEK, Ultem, Delrin), and composite materials (carbon fiber pre-preg, fiberglass).

PRICING: Metals: index-based pricing with monthly adjustment. Published base price plus Redwood premium of 8%. Index: LME (London Metal Exchange) for aluminum, MEPS for stainless steel. Plastics: fixed quarterly pricing with 60 days advance notice of changes. Volume pricing: annual purchases exceeding USD 500,000 qualify for 5% rebate.

QUALITY AND CERTIFICATION: Mill test reports (MTRs) or certificates of conformance provided with every shipment. Material traceability maintained from mill to delivery. Shelf life documentation for plastics and composites.

DELIVERY: Standard lead time: 2-4 weeks depending on material and quantity. Stock items available for same-week shipment. Consignment inventory program available for high-volume materials (minimum USD 50,000 inventory value maintained at Buyer facility).

FORCE MAJEURE: Supply disruptions due to mine closures, trade restrictions, or natural disasters. Redwood Materials Corp maintains 90-day safety stock of critical materials. Alternative sourcing assistance provided during supply disruptions. Governing law: State of California, United States.'),

('VC-038', 25, 'Lone Star Industrial - Heavy Equipment Lease and Service', '2024-01-01', '2026-12-31',
'HEAVY EQUIPMENT LEASE AND SERVICE AGREEMENT between Buyer and Lone Star Industrial.

PAYMENT TERMS: NET30 from monthly invoice date. Monthly lease payment: USD 45,000 for equipment fleet. Service and maintenance billed separately at cost plus 15%.

EQUIPMENT: 10 forklifts (5,000 lb capacity), 3 overhead cranes (10-ton capacity), 2 CNC lathes, 1 5-axis CNC mill, and associated tooling. All equipment current model year or 1 year prior.

MAINTENANCE: Full preventive maintenance program per manufacturer specifications. Monthly PM visits included. Emergency breakdown response: 4 hours during business hours, 8 hours after hours. Loaner equipment provided if repair exceeds 48 hours.

OPERATOR TRAINING: Lone Star Industrial provides OSHA-compliant operator training for all leased equipment. Initial training at contract start (up to 30 operators). Refresher training annually. New employee training within 5 business days of request.

UTILIZATION REPORTING: Monthly equipment utilization reports showing hours of operation, maintenance history, and fuel/energy consumption. Fleet optimization recommendations quarterly.

INSURANCE: Lone Star Industrial maintains comprehensive equipment insurance. Buyer responsible for deductible (USD 5,000 per incident). Damage from operator negligence or unauthorized modifications billed to Buyer at repair cost. Governing law: State of Texas, United States.'),

('VC-039', 26, 'Cascade Components Inc - Electronic Components Supply', '2024-08-01', '2026-07-31',
'ELECTRONIC COMPONENTS SUPPLY AGREEMENT between Buyer and Cascade Components Inc.

PAYMENT TERMS: NET30 from invoice date. All amounts in USD.

PRODUCT SCOPE: Electronic components including: passive components (resistors, capacitors, inductors), semiconductors (ICs, transistors, diodes), connectors and cables, PCB assemblies, and custom cable harnesses.

PRICING: Catalog components: published distributor pricing less 12%. Custom assemblies: quoted per BOM (Bill of Materials) with 10 business day quote turnaround. Annual volume rebate: 3% on purchases exceeding USD 250,000.

LEAD TIMES: Stock components: same-day ship for orders by 2pm PST. Non-stock catalog: 2-4 weeks. Custom assemblies: 4-6 weeks after BOM approval. Critical shortage escalation: Cascade maintains allocation agreements with tier-1 manufacturers.

QUALITY: IPC-A-610 Class 2 (or Class 3 upon request) for PCB assemblies. Component authenticity guaranteed (AS6171 compliant). Counterfeit parts prevention program with 100% incoming inspection for high-risk components.

OBSOLESCENCE MANAGEMENT: Cascade monitors end-of-life notifications for all components in Buyer BOMs. 12-month advance notice of last-time-buy opportunities. Alternative component suggestions provided within 10 business days of EOL notification. Governing law: State of Washington, United States.'),

('VC-040', 27, 'Great Lakes Tooling - Precision Tooling Contract', '2025-04-01', '2027-03-31',
'PRECISION TOOLING CONTRACT between Buyer and Great Lakes Tooling.

PAYMENT TERMS: NET45 from delivery and acceptance. Tooling projects exceeding USD 50,000: progress payments at 25% milestones.

SERVICES: Custom tooling design and manufacture including: injection molds, die cast dies, stamping dies, jigs and fixtures, and gauge and inspection tooling. Materials: tool steel (H13, S7, A2, D2), aluminum tooling for prototype and short-run production.

DESIGN PROCESS: Great Lakes Tooling provides DFM (Design for Manufacturability) analysis within 5 business days of receiving part models. 3D tool design review at 50% and 90% completion. First article samples within 3 business days of tool completion.

TOOL LIFE GUARANTEE: Injection molds: 500,000 shots minimum for production steel tools. Stamping dies: 1,000,000 hits minimum. Great Lakes Tooling repairs or replaces tooling that fails to meet guaranteed life at no cost (normal wear conditions).

STORAGE: Great Lakes Tooling stores all active tooling at no charge. Inactive tooling (no orders in 18 months): USD 150/month storage fee. Tooling destruction only with written Buyer authorization.

MODIFICATIONS: Engineering changes quoted within 5 business days. Typical modification lead time: 2-4 weeks depending on complexity. Rush modifications at 30% premium with 1-week lead time. Governing law: State of Michigan, United States.');

In [ ]:
-- Insert vendor contracts (batch 7: Additional contracts for multi-doc scenarios)
INSERT INTO COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACTS VALUES
('VC-041', 22, 'Summit Manufacturing Inc - Quality Assurance Addendum', '2025-01-01', '2027-01-31',
'QUALITY ASSURANCE ADDENDUM to Contract Manufacturing Agreement between Buyer and Summit Manufacturing Inc.

PURPOSE: This addendum establishes enhanced quality requirements for aerospace and defense program parts manufactured by Summit Manufacturing Inc.

ADDITIONAL QUALITY REQUIREMENTS: All parts for Programs A, B, and C require AS9100D compliance with additional customer flow-down requirements. First Article Inspection (FAI) per AS9102 for all new and changed parts. Statistical Process Control (SPC) on all critical characteristics with Cpk minimum 1.67.

NONCONFORMANCE MANAGEMENT: Summit Manufacturing implements a formal nonconformance process. NCRs documented within 24 hours of discovery. Root cause analysis (8D methodology) completed within 10 business days. Corrective action verification within 30 days.

AUDIT RIGHTS: Buyer reserves the right to perform on-site quality audits with 5 business days notice. Annual audit required; additional audits triggered by quality escapes. Summit Manufacturing provides access to all quality records and production areas.

TRACEABILITY: Full material and process traceability from raw material to finished part. Lot and serial number tracking per Buyer specification. Records retained minimum 10 years.'),

('VC-042', 30, 'Beacon Consulting Group - Data Analytics SOW', '2025-06-01', '2025-12-31',
'STATEMENT OF WORK: DATA ANALYTICS TRANSFORMATION under MSA between Buyer and Beacon Consulting Group.

PROJECT SCOPE: Beacon Consulting Group will lead a 7-month engagement to transform Buyers data analytics capabilities. Key workstreams: (1) Data governance framework design, (2) Self-service BI platform implementation, (3) Advanced analytics use case development, (4) Center of Excellence establishment.

TEAM: 1 Partner (10% allocation), 1 Senior Manager (50%), 2 Senior Consultants (100%), 1 Consultant (100%). Total estimated hours: 4,200.

BUDGET: Fixed price: USD 1,050,000. Milestone payments: Kickoff 15% (USD 157,500), Data governance framework delivery 25% (USD 262,500), BI platform go-live 35% (USD 367,500), Final deliverables and CoE handoff 25% (USD 262,500).

KEY DELIVERABLES: Data governance policy document and RACI matrix. Implemented BI platform with 15 executive dashboards. Three production ML models for demand forecasting, customer churn, and pricing optimization. CoE operating model with training curriculum for 50 internal analysts.

ACCEPTANCE CRITERIA: Each deliverable has specific acceptance criteria in Appendix B. Buyer has 10 business days to accept or provide feedback. Maximum 2 rounds of revision per deliverable.'),

('VC-043', 16, 'Dublin Cloud Ltd - Data Sovereignty Addendum', '2025-06-01', '2026-12-31',
'DATA SOVEREIGNTY ADDENDUM to Cloud Infrastructure Contract between Buyer and Dublin Cloud Ltd.

PURPOSE: This addendum strengthens data residency and sovereignty requirements for Buyers expansion into regulated industries (financial services, healthcare).

DATA CLASSIFICATION: Tier 1 (Restricted): personal data, financial records, health information. Must reside exclusively in Ireland data center. No cross-border processing or replication. Tier 2 (Confidential): business-critical but non-regulated data. EU-only residency required. Netherlands DR site permitted. Tier 3 (Internal): non-sensitive operational data. Standard EU residency policy applies.

ENCRYPTION: Tier 1 data: customer-managed keys (BYOK) mandatory. Key rotation every 90 days. Hardware Security Module (HSM) backed. Tier 2: Dublin Cloud Ltd managed keys with annual rotation. All tiers: TLS 1.3 minimum for data in transit.

ACCESS CONTROLS: Tier 1 data accessible only by named individuals approved by Buyer Data Protection Officer. Dublin Cloud Ltd staff access requires 48-hour advance approval and is logged with full audit trail. No sub-processor access to Tier 1 data without specific written authorization.

BREACH NOTIFICATION: Dublin Cloud Ltd notifies Buyer within 4 hours of discovering a confirmed data breach affecting Buyer data. Preliminary incident report within 24 hours. Full incident report with root cause within 72 hours. Compliance with GDPR Article 33 notification timelines.

AUDIT: Buyer may conduct data sovereignty audits annually. Dublin Cloud Ltd provides SOC 2 Type II and ISO 27001 certificates. Independent GDPR compliance audit performed annually by Big 4 firm, results shared with Buyer.'),

('VC-044', 9, 'Paris Conseil SARL - Change Management Engagement', '2025-09-01', '2026-06-30',
'STATEMENT OF WORK: ORGANIZATIONAL CHANGE MANAGEMENT under Framework Agreement between Buyer and Paris Conseil SARL.

PROJECT: Support organizational change management for Buyers ERP system migration from legacy platforms to cloud-based solution.

TEAM: 1 Managing Director (20% oversight), 2 Senior Consultants (100% dedicated), 1 Consultant (100% dedicated). Estimated duration: 10 months. Total estimated hours: 5,600.

BUDGET: T&M engagement estimated at EUR 1,176,000 based on blended rate of EUR 210/hour. Monthly cap: EUR 135,000 unless written approval for overage obtained in advance.

DELIVERABLES: Stakeholder analysis and change impact assessment. Communication plan and collateral (town halls, newsletters, FAQs in 4 languages). Training needs assessment and curriculum design. Train-the-trainer program for 25 internal change champions. Post-go-live adoption monitoring dashboard with weekly pulse surveys.

TRAVEL: Up to 4 consultants traveling to Buyers EU sites (Germany, Netherlands, France, Spain) monthly. Travel costs estimated at EUR 8,000/month, billed at cost within the approved cap.'),

('VC-045', 38, 'Northstar Manufacturing - Prototype Development Agreement', '2025-03-01', '2026-08-31',
'PROTOTYPE DEVELOPMENT AGREEMENT between Buyer and Northstar Manufacturing.

PAYMENT TERMS: NET30. Prototype projects: 40% deposit upon project approval, 30% upon design completion, 30% upon prototype delivery and acceptance.

SERVICES: Rapid prototyping services including: 3D printing (SLA, SLS, FDM), CNC prototype machining, vacuum casting for short-run production (up to 50 units), and functional prototype assembly and testing.

LEAD TIMES: 3D printed prototypes: 3-5 business days. CNC machined prototypes: 7-10 business days. Vacuum cast parts: 15-20 business days (including silicone mold creation). Functional assembly: add 5-10 business days depending on complexity.

DESIGN SUPPORT: Northstar provides DFM feedback within 2 business days of receiving 3D models. Tolerance analysis and material recommendation services included at no additional charge for prototype projects. Reverse engineering services available at USD 150/hour.

CONFIDENTIALITY: All prototype designs, specifications, and test results are Buyer confidential. Northstar maintains separate locked storage for Buyer prototype materials. NDA in effect for 5 years post-contract. Northstar staff working on Buyer prototypes sign individual NDAs.

QUALITY: Dimensional inspection report with every prototype delivery. Material certification for metal prototypes. Functional test results per Buyer test plan (if provided). Governing law: State of Minnesota, United States.'),

('VC-046', 20, 'Helsinki Data Oy - Professional Services Addendum', '2025-07-01', '2027-03-31',
'PROFESSIONAL SERVICES ADDENDUM to Data Platform Services Agreement between Buyer and Helsinki Data Oy.

SCOPE: Helsinki Data Oy provides professional services for data platform implementation, migration, and optimization projects beyond the base platform subscription.

RATE CARD: Data Architect: EUR 220/hour. Senior Data Engineer: EUR 185/hour. Data Engineer: EUR 145/hour. ML Engineer: EUR 200/hour. Project Manager: EUR 160/hour. Rates fixed for duration of this addendum.

ENGAGEMENT MODEL: Projects initiated via Statement of Work under this addendum. Fixed-price and T&M models available. Minimum engagement: 80 hours. Helsinki Data Oy provides project estimates within 5 business days of receiving requirements.

KNOWLEDGE TRANSFER: All projects include minimum 20% knowledge transfer time. Comprehensive documentation per Helsinki Data Oy standards (architecture docs, runbooks, troubleshooting guides). Code review and walkthrough sessions with Buyer technical team.

SUPPORT TRANSITION: Post-project support: 30 days of bug fixes included at no additional cost. Extended hyper-care available at 50% of project rates for 60 days following project completion. Issues identified after hyper-care period handled under the base platform support SLA.'),

('VC-047', 6, 'Global Logistik BV - Cross-Border Trade Compliance', '2025-04-01', '2026-12-31',
'CROSS-BORDER TRADE COMPLIANCE ADDENDUM to Freight Services Agreement between Buyer and Global Logistik BV.

SCOPE: Global Logistik BV provides customs brokerage and trade compliance services for Buyers EU import/export activities.

CUSTOMS BROKERAGE: Preparation and filing of customs declarations for all Buyer shipments crossing EU external borders. Classification advisory (HS/CN codes). Preferential origin determination and management of EUR.1 and ATR certificates.

COMPLIANCE: Global Logistik BV maintains AEO-F (Authorised Economic Operator - Full) certification. Annual compliance review of Buyers trade activities. Export control screening against EU consolidated sanctions list and US BIS Entity List. Denied party screening for all consignees.

FEES: Standard customs declaration: EUR 45 per filing. Complex declarations (multiple commodity codes, special procedures): EUR 85 per filing. Trade compliance advisory: EUR 150/hour. Annual compliance audit: EUR 5,000.

DUTY OPTIMIZATION: Global Logistik BV actively manages customs duty exposure. Services include: tariff engineering recommendations, free trade agreement utilization analysis (targeting 90%+ utilization on eligible trade flows), customs warehouse and inward processing relief applications.'),

('VC-048', 29, 'Metro IT Solutions LLC - Cloud Migration SOW', '2025-05-01', '2026-02-28',
'STATEMENT OF WORK: CLOUD MIGRATION PROJECT under IT Support Services Contract between Buyer and Metro IT Solutions LLC.

PROJECT SCOPE: Migrate 45 on-premises applications to cloud infrastructure. Phase 1 (months 1-3): assessment and planning for all 45 applications. Phase 2 (months 4-7): migrate 30 lift-and-shift applications. Phase 3 (months 8-10): re-architect and migrate 15 complex applications.

TEAM: 1 Cloud Architect (100%), 2 Senior Cloud Engineers (100%), 3 Cloud Engineers (100%), 1 Project Manager (50%). Total estimated hours: 12,800.

BUDGET: Fixed price: USD 2,560,000. Milestone payments aligned with phase completion: Phase 1 complete: 20%. Phase 2 complete: 45%. Phase 3 complete: 25%. Hypercare and closeout: 10%.

SUCCESS CRITERIA: All 45 applications operational in cloud with zero data loss. Application performance equal to or better than on-premises baseline. Total cloud infrastructure cost within 15% of agreed target. Zero unplanned downtime during migration weekends.

ROLLBACK: Each application migration includes a documented rollback plan. Rollback window: 48 hours post-migration. Metro IT Solutions maintains on-premises environment operational for 30 days post each migration wave as fallback.'),

('VC-049', 25, 'Lone Star Industrial - Safety Equipment Addendum', '2025-02-01', '2026-12-31',
'SAFETY EQUIPMENT ADDENDUM to Heavy Equipment Lease and Service Agreement between Buyer and Lone Star Industrial.

SCOPE: Lone Star Industrial provides additional safety equipment and compliance services for Buyers manufacturing facilities.

EQUIPMENT: 50 personal gas monitors (4-gas: LEL, O2, CO, H2S), 20 fall protection harness kits with self-retracting lifelines, 10 confined space tripod rescue systems, emergency eyewash and shower stations (15 units), fire suppression equipment maintenance.

PRICING: Monthly safety equipment lease: USD 8,500. Annual OSHA compliance audit: USD 12,000. Safety training per session (up to 25 attendees): USD 2,500. Equipment calibration and certification: included in monthly lease.

COMPLIANCE: All equipment meets or exceeds OSHA, ANSI, and NFPA standards. Monthly inspection and calibration of all gas monitors. Annual load testing of fall protection equipment. Lone Star Industrial maintains calibration records for minimum 5 years.

TRAINING: Lone Star Industrial provides quarterly safety training sessions covering: confined space entry (OSHA 1910.146), fall protection (OSHA 1926.502), hazard communication (OSHA 1910.1200), and emergency response procedures. Training records maintained and available for OSHA inspection.'),

('VC-050', 10, 'Lyon Industrie SA - Supply Chain Resilience Addendum', '2025-06-01', '2027-12-31',
'SUPPLY CHAIN RESILIENCE ADDENDUM to Manufacturing Parts Agreement between Buyer and Lyon Industrie SA.

PURPOSE: Establish dual-sourcing and supply chain risk mitigation measures in response to recent supply chain disruptions.

SAFETY STOCK: Lyon Industrie SA maintains 60 days of safety stock for all A-class parts (top 20% by annual spend). B-class parts: 30 days. C-class parts: standard lead time. Safety stock costs shared 50/50 between Buyer and Lyon Industrie SA.

DUAL SOURCING: Lyon Industrie SA qualifies and maintains a secondary manufacturing site (Romania facility) capable of producing 100% of Buyer part numbers within 30 days of activation. Annual qualification audit of secondary site by Buyer.

RISK MONITORING: Weekly supply chain risk dashboard covering: raw material availability, logistics disruptions, geopolitical risk scores, and supplier financial health. Automated alerts when risk scores exceed predetermined thresholds.

BUSINESS CONTINUITY: In the event of a declared force majeure at the primary Lyon facility, the Romanian backup site activates within 72 hours. Lyon Industrie SA maintains business interruption insurance of EUR 20,000,000. Buyer invoicing continues at standard pricing during backup site operations (no surcharge for first 90 days).

COST: Supply chain resilience surcharge: 3% on all part prices under this addendum. Surcharge reviewed annually and adjusted based on actual safety stock carrying costs and secondary site maintenance expenses. Governing law: France.');

In [ ]:
-- Verify: should show 50 rows across ~25 vendors
SELECT COUNT(*) AS total_contracts,
       COUNT(DISTINCT VENDOR_ID) AS unique_vendors,
       MIN(EFFECTIVE_DATE) AS earliest_contract,
       MAX(EXPIRATION_DATE) AS latest_expiration
FROM COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACTS;

## Cortex Search Service

Create a search service over the contract content so the agent can find relevant contract language by natural language query. The `VENDOR_ID` attribute enables filtered search (e.g., search only within a specific vendor's contracts).

In [ ]:
-- Create Cortex Search Service over vendor contracts
CREATE OR REPLACE CORTEX SEARCH SERVICE COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACT_SEARCH
  ON CONTENT
  ATTRIBUTES VENDOR_ID
  WAREHOUSE = DEFAULT_WH
  TARGET_LAG = '1 hour'
  AS (
    SELECT
      DOC_ID,
      VENDOR_ID,
      CONTRACT_TITLE,
      CONTENT
    FROM COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACTS
  );

In [ ]:
-- Verify: search for "uptime SLA penalty" — should return Helsinki Data Oy, Frankfurt IT, Dublin Cloud
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACT_SEARCH',
    '{
      "query": "uptime SLA penalty for downtime",
      "columns": ["DOC_ID", "CONTRACT_TITLE"],
      "limit": 5
    }'
  )
) AS results;

## Agent Update Instructions

After running all cells above, manually add the search tool to `AP_ANALYTICS_ASSISTANT`. You can do this via Cortex Code:

```
$agent-studio Edit AP_ANALYTICS_ASSISTANT: add a cortex_search tool
called "contract_search" pointing to VENDOR_CONTRACT_SEARCH.
```

Or apply this YAML spec change directly:

### Add to `tools` array:
```yaml
- tool_spec:
    type: cortex_search
    name: contract_search
    description: >-
      Search vendor contract documents for payment terms, SLAs, penalties,
      volume discounts, renewal clauses, and legal provisions. Use when the
      user asks about what a contract says, what terms were agreed, or
      contractual obligations for a specific vendor.
```

### Add to `tool_resources`:
```yaml
contract_search:
  search_service: COCO_WORKSHOP.ANALYTICS.VENDOR_CONTRACT_SEARCH
  id_column: DOC_ID
  title_column: CONTRACT_TITLE
  max_results: 5
```

### Update `instructions.orchestration`:
```yaml
orchestration: >-
  Use ap_data for questions about invoice amounts, counts, spend, balances, and payment history.
  Use contract_search for questions about contract terms, SLAs, penalties, discounts, and legal clauses.
  When a question involves both (e.g., "is this vendor meeting their SLA?" or "are we paying the contracted rate?"), use both tools.
```

## Demo Prompts

After updating the agent, try these prompts. They're ordered from simple (single tool) to impressive (multi-tool hybrid).

### Search-Only (contract lookup)
1. **"What is the uptime SLA for Helsinki Data Oy?"**
   → Should find 99.99% uptime guarantee with 2% credit per hour of downtime

2. **"What are the penalty clauses in our Siemens contract?"**
   → Should surface the 3% quarterly penalty for late delivery and 5% credit for defects

3. **"Which vendor contracts have auto-renewal clauses?"**
   → Should find Siemens (120-day notice), Helsinki Data (60-day notice), Beacon Consulting (90-day notice), and others

4. **"What is Beacon Consulting Group's hourly rate for a Senior Consultant?"**
   → Should find USD 275/hour from the MSA rate card

### Hybrid (SQL + Search — the showstoppers)
5. **"Are we paying Siemens AG within their contracted payment terms?"**
   → Agent searches contract (finds NET60), then queries AP_INVOICES (finds all four terms used). Highlights the mismatch.

6. **"Pacific Freight has a volume discount at $50K/month. Are we hitting that threshold?"**
   → Agent searches contract (finds 12% discount at USD 50K+/month), then queries monthly spend for Pacific Freight.

7. **"Which of our top 5 vendors by spend have contracts expiring in 2026?"**
   → Agent runs SQL for top vendors by spend, then searches contracts for expiration dates.

8. **"What is Helsinki Data Oy's SLA penalty, and how much do we spend with them annually?"**
   → Agent searches contract (2% per hour of downtime, 99.99% uptime), then queries total spend.

9. **"Compare the contracted payment terms for our German vendors against what we're actually paying."**
   → Agent searches contracts for Siemens, Bosch, Rheinland, Muller, Frankfurt IT payment terms, then queries AP_INVOICES for actual terms used per vendor.

10. **"What safety and compliance certifications do our IT services vendors contractually guarantee?"**
    → Agent searches IT vendor contracts for ISO 27001, SOC 2, GDPR references across Frankfurt IT, Dublin Cloud, Helsinki Data, Metro IT, Pinnacle IT.